# Probability and Statistics Mastery Lab

Compare two systems on shared items. The lab makes the estimand, pairing, bootstrap interval, randomization test, Bayesian marginal update, and sample-size behavior explicit.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(15)
np.set_printoptions(precision=4, suppress=True)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


## 1. Simulate paired evaluation data

Item difficulty affects both systems, creating the dependence that a paired design should retain.


In [ ]:
n = 600
difficulty = rng.normal(size=n)
group = rng.integers(0, 2, size=n)
p_a = sigmoid(0.35 - difficulty)
p_b = sigmoid(0.45 - difficulty + 0.20*group)
score_a = (rng.random(n) < p_a).astype(float)
score_b = (rng.random(n) < p_b).astype(float)
diff = score_b - score_a

estimand = "Expected paired accuracy difference B-A on this item-generating distribution"
print(estimand)
print("A accuracy:", score_a.mean(), "B accuracy:", score_b.mean(), "difference:", diff.mean())
assert set(np.unique(diff)).issubset({-1.0, 0.0, 1.0})


## 2. Analytic paired interval

This large-sample interval uses the sample SD of per-item differences. It is an approximation, not a guarantee for every data-generating process.


In [ ]:
estimate = diff.mean()
paired_se = diff.std(ddof=1) / np.sqrt(n)
paired_ci = estimate + np.array([-1,1]) * 1.96 * paired_se
unpaired_se = np.sqrt(score_a.var(ddof=1)/n + score_b.var(ddof=1)/n)

print("paired SE:", paired_se, "unpaired SE:", unpaired_se)
print("approximate 95% paired CI:", paired_ci)
assert paired_ci[0] <= estimate <= paired_ci[1]


## 3. Paired bootstrap

Resample item indices and keep each A/B pair together.


In [ ]:
def paired_bootstrap(a, b, draws=3000, seed=0):
    local = np.random.default_rng(seed)
    n = len(a)
    indices = local.integers(0, n, size=(draws, n))
    return (b[indices] - a[indices]).mean(axis=1)

boot = paired_bootstrap(score_a, score_b, seed=151)
boot_ci = np.quantile(boot, [0.025, 0.975])
print("bootstrap CI:", boot_ci)

plt.hist(boot, bins=40, density=True)
plt.axvline(estimate, color="black", label="observed")
plt.axvline(boot_ci[0], color="red", linestyle="--")
plt.axvline(boot_ci[1], color="red", linestyle="--", label="95% interval")
plt.xlabel("paired mean difference"); plt.ylabel("density"); plt.legend();


## 4. Paired randomization test

Under the sharp exchangeability null, swapping labels A/B within each pair is equivalent to random sign flips of the differences.


In [ ]:
def paired_randomization_pvalue(diff, draws=10000, seed=0):
    local = np.random.default_rng(seed)
    signs = local.choice([-1.0, 1.0], size=(draws, len(diff)))
    null_stats = (signs * diff).mean(axis=1)
    observed = diff.mean()
    return (1 + np.sum(np.abs(null_stats) >= abs(observed))) / (draws + 1)

p_value = paired_randomization_pvalue(diff, seed=152)
print("two-sided paired randomization p-value:", p_value)
assert 0 <= p_value <= 1


## 5. Beta-Binomial marginal updates

These posteriors quantify each marginal success rate. They do not by themselves model the A/B pairing.


In [ ]:
prior_alpha = prior_beta = 1.0
post_a = np.array([prior_alpha + score_a.sum(), prior_beta + n - score_a.sum()])
post_b = np.array([prior_alpha + score_b.sum(), prior_beta + n - score_b.sum()])
mean_a = post_a[0] / post_a.sum()
mean_b = post_b[0] / post_b.sum()
print("posterior mean A:", mean_a, "posterior mean B:", mean_b)
assert abs(mean_a - score_a.mean()) < 0.002
assert abs(mean_b - score_b.mean()) < 0.002


## 6. Subgroups and distribution shift

Treat subgroup inspection as exploratory unless it was predeclared. A changed item distribution changes the estimand.


In [ ]:
for g in [0,1]:
    mask = group == g
    d = (score_b[mask]-score_a[mask])
    print("group", g, "n", mask.sum(), "difference", d.mean(), "SE", d.std(ddof=1)/np.sqrt(mask.sum()))

shifted_weights = np.where(group == 1, 3.0, 1.0)
shifted_estimate = np.average(diff, weights=shifted_weights)
print("original estimate:", estimate, "shift-weighted estimate:", shifted_estimate)


## Exercises

1. Repeat the bootstrap incorrectly by resampling A and B independently. Explain the uncertainty change.
2. Simulate a true null 500 times and inspect the randomization-test p-value distribution.
3. Estimate bootstrap coverage for a known simulated effect.
4. Predeclare a meaningful effect and simulate power at n=100, 300, 1,000, and 3,000.
5. Add five secondary metrics and compare Bonferroni correction with false-discovery-rate control.
6. Explain why separate Beta posteriors do not fully answer the paired-difference question.

Carry the completed notebook into the Paired Model Evaluation Under Uncertainty capstone.
